# Indirect losses: cascading losses and temporal-ratio parameter

Reproduces the **left panel of `cascading_multi-basin_scenarios_v2.png`** (main text, `fig:res3`). The right panel of that figure is a hand-drawn timing schematic.

**τ = time_lag / duration** — the lag between the onsets of successive basin disruptions, normalised by their duration: τ = 0 simultaneous, τ = 1 back to back, τ > 1 separated by a recovery interval.

**Data**: the two cascading sweeps `data/monte_carlo/cambodia_simulations_results_230927_cascading.csv` and `cambodia_simulations_results_230927_cascading_v2.csv`, merged.

**Environment**: `dsc` conda env.

In [1]:
from pathlib import Path

BUNDLE = Path.cwd()
MC = BUNDLE / "data" / "monte_carlo"
SCEN = BUNDLE / "data" / "scenarios"
REF = BUNDLE / "data" / "reference"
FIGDIR = BUNDLE / "figures" / "generated"
FIGDIR.mkdir(parents=True, exist_ok=True)


def save_plotly(fig, name):
    """Interactive HTML always; PNG at the layout size if the `kaleido` package is installed."""
    fig.write_html(FIGDIR / f"{name}.html", include_plotlyjs="cdn")
    try:
        fig.write_image(FIGDIR / f"{name}.png")
    except Exception:
        pass

## Setup and data

In [2]:
# All imports
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Color scheme for consistency
colors = {
    'single_basin_mekong': '#1f77b4',
    'single_basin_saigon': '#ff7f0e', 
    'single_basin_tonle_sap': '#2ca02c',
    'cascading_hazards_0d': '#d62728',
    'sum_single_basins': '#9467bd'
}

In [3]:
# Load and merge cascading hazards datasets
df_cascading_1 = pd.read_csv(MC / 'cambodia_simulations_results_230927_cascading.csv')
df_cascading_2 = pd.read_csv(MC / 'cambodia_simulations_results_230927_cascading_v2.csv')

print("="*80)
print("CASCADING HAZARDS DATA")
print("="*80)
print(f"\nOriginal cascading dataset: {df_cascading_1.shape[0]} rows")
print(f"Cascading v2 dataset: {df_cascading_2.shape[0]} rows")

# Merge the datasets
df_cascading = pd.concat([df_cascading_1, df_cascading_2], ignore_index=True)
print(f"Merged cascading dataset: {df_cascading.shape[0]} rows")

# Check available time_lag and duration combinations
combinations = df_cascading.groupby(['duration', 'time_lag']).size().reset_index(name='count')
combinations['tau'] = combinations['time_lag'] / combinations['duration']
print(f"\nAvailable combinations: {len(combinations)}")
print(f"Duration range: {combinations['duration'].min()} - {combinations['duration'].max()} days")
print(f"Tau range: {combinations['tau'].min():.2f} - {combinations['tau'].max():.2f}")
print("\n" + "="*80)

CASCADING HAZARDS DATA

Original cascading dataset: 1710 rows
Cascading v2 dataset: 820 rows
Merged cascading dataset: 2530 rows

Available combinations: 50
Duration range: 2 - 12 days
Tau range: 0.00 - 3.00



In [4]:
def map_time_lag_to_tau(time_lag, duration):
    """
    Map time_lag to Tau based on disruption duration.
    τ represents the normalized temporal separation between events.
    τ = time_lag / duration
    """
    return time_lag / duration

def get_y_range(data_list, extend_below=0.5):
    """
    Calculate y-axis range from list of dataframes with consistent padding.
    """
    all_means = pd.concat([df['mean'] for df in data_list])
    all_stds = pd.concat([df['std'] for df in data_list])
    y_min = (all_means - all_stds).min()
    y_max = (all_means + all_stds).max()
    padding = 0.1 * (y_max - y_min)
    return [y_min - padding - extend_below, y_max + padding]

Mean and standard deviation of household losses per τ, for each disruption duration.

In [5]:
# Define scenarios with extended τ values based on available data
scenarios_cascading = [
    {
        'duration': 2,
        'time_lags': [0, 1, 2, 3, 4, 5, 6],  # τ = [0, 0.5, 1, 1.5, 2, 2.5, 3]
        'name': '2-day disruption',
        'color': '#1f77b4'
    },
    {
        'duration': 4,
        'time_lags': [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],  # τ = [0, 0.25, 0.5, 0.75, 1, ...]
        'name': '4-day disruption',
        'color': '#ff7f0e'
    },
    {
        'duration': 8,
        'time_lags': [0, 2, 4, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24],  # τ = [0, 0.25, 0.5, 0.75, 1, ...]
        'name': '8-day disruption',
        'color': '#2ca02c'
    },
    {
        'duration': 12,
        'time_lags': [0, 3, 6, 9, 12, 15, 18, 21, 24, 27, 30, 33, 36],  # τ = [0, 0.25, 0.5, 0.75, 1, ...]
        'name': '12-day disruption',
        'color': '#d62728'
    }
]

# Create scenario data with extended τ values
scenario_data_cascading = []
for scenario in scenarios_cascading:
    df_filtered = df_cascading[df_cascading['duration'] == scenario['duration']].copy()
    df_filtered = df_filtered[df_filtered['time_lag'].isin(scenario['time_lags'])]
    
    # Add Tau column
    df_filtered['tau'] = df_filtered.apply(
        lambda row: map_time_lag_to_tau(row['time_lag'], row['duration']), axis=1
    )
    
    # Calculate statistics by tau
    stats_df = df_filtered.groupby(['duration', 'tau'])['household_loss'] \
                         .agg(['mean', 'std']).reset_index()
    
    scenario_data_cascading.append({
        'stats_df': stats_df,
        'name': scenario['name'],
        'color': scenario['color']
    })

# Print available tau values by scenario
print("Available τ values by scenario:")
for scenario in scenario_data_cascading:
    available_taus = sorted(scenario['stats_df']['tau'].unique())
    print(f"  {scenario['name']}: {[f'{t:.2f}' for t in available_taus]}")

Available τ values by scenario:
  2-day disruption: ['0.00', '0.50', '1.00', '1.50', '2.00', '2.50', '3.00']
  4-day disruption: ['0.00', '0.25', '0.50', '0.75', '1.00', '1.25', '1.50', '1.75', '2.00', '2.25', '2.50', '2.75', '3.00']
  8-day disruption: ['0.00', '0.25', '0.50', '0.75', '1.00', '1.25', '1.50', '1.75', '2.00', '2.25', '2.50', '2.75', '3.00']
  12-day disruption: ['0.00', '0.25', '0.50', '0.75', '1.00', '1.25', '1.50', '1.75', '2.00', '2.25', '2.50', '2.75', '3.00']


### Cascading multi-basin scenarios

sampling 50 MC simulations is just because for some durations and tau combinations I did more simulations, so I leveraged everything to have 50 for every point.

In [6]:
_grid_pairs = {(s['duration'], lag) for s in scenarios_cascading for lag in s['time_lags']}
_mask = df_cascading.apply(lambda r: (r['duration'], r['time_lag']) in _grid_pairs, axis=1)

df_cascading_n50 = (
    df_cascading[_mask]
    .sort_values(['duration', 'time_lag', 'filename', 'mc_repetition'])
    .groupby(['duration', 'time_lag'], sort=False)
    .head(50)
    .reset_index(drop=True)
)

_counts = df_cascading_n50.groupby(['duration', 'time_lag']).size()
assert (_counts == 50).all(), f"Unbalanced after subsampling: {_counts[_counts != 50].to_dict()}"

# Sampling 50 MC simulations
scenario_data_n50 = []
for scenario in scenarios_cascading:
    df_filtered = df_cascading_n50[df_cascading_n50['duration'] == scenario['duration']].copy()
    df_filtered = df_filtered[df_filtered['time_lag'].isin(scenario['time_lags'])]
    df_filtered['tau'] = df_filtered['time_lag'] / df_filtered['duration']
    stats_df = (df_filtered
                .groupby(['duration', 'tau'])['household_loss']
                .agg(['mean', 'std'])
                .reset_index())
    scenario_data_n50.append({
        'stats_df': stats_df,
        'name': scenario['name'],
        'color': scenario['color']
    })

# Keep only 4/8/12-day scenarios
keep = {'4-day disruption', '8-day disruption', '12-day disruption'}
scenarios_n50 = [s for s in scenario_data_n50 if s['name'] in keep]

fig = go.Figure()
for s in scenarios_n50:
    data = s['stats_df'].sort_values('tau')
    fig.add_trace(
        go.Scatter(
            x=data['tau'],
            y=data['mean'],
            error_y=dict(type='data', array=data['std'], visible=True),
            mode='lines+markers',
            name=s['name'],
            line=dict(width=3, color=s['color']),
            marker=dict(size=8, color=s['color'])
        )
    )

fig.update_layout(
    width=1200,
    height=900,
    template='plotly_white',
    margin=dict(l=90, r=40, t=60, b=80),
    legend=dict(orientation='h', x=0.02, y=1.05, font=dict(size=24))
)
fig.update_xaxes(
    title_text='τ (temporal ratio parameter)',
    tickmode='array',
    tickvals=[0, 0.5, 1, 1.5, 2, 2.5, 3],
    ticktext=['0', '0.5', '1', '1.5', '2', '2.5', '3'],
    range=[-0.1, 3.1],
    title_font=dict(size=24),
    tickfont=dict(size=20),
)
ymin = min(df['mean'].min() - df['std'].max() for df in [s['stats_df'] for s in scenarios_n50])
ymax = max(df['mean'].max() + df['std'].max() for df in [s['stats_df'] for s in scenarios_n50])
fig.update_yaxes(
    title_text='Indirect Losses [M$]',
    range=[max(0, ymin), ymax],
    title_font=dict(size=24),
    tickfont=dict(size=20)
)

save_plotly(fig, 'cascading_multi-basin_scenarios_v2_left_panel_n50')
fig.show()


/var/folders/q5/j53bmmsn5_v21fb0ltm6q7w40000gn/T/ipykernel_90408/766761587.py:15: DeprecationWarning:


Support for Kaleido versions less than 1.0.0 is deprecated and will be removed after September 2025.
Please upgrade Kaleido to version 1.0.0 or greater (`pip install 'kaleido>=1.0.0'` or `pip install 'plotly[kaleido]'`).


